# Zero-Train Optimization & Maintenance (ZTOM) of a Legal Understanding LLM

**Legal Understanding LLM (Gemma3-1B + LoRA adapters)**

## Background

The Medical Chatbot is a large language model (LLM) based on Llama3.2 with LoRA adapters that was trained on online medical forums to provide medical advice from healthcare-related queries. The model processes medical questions and provides appropriate responses based on patterns learned from medical forum discussions.

**Dataset:** https://huggingface.co/datasets/lextale/FirstAidInstructionsDataset/viewer/default/icliniqDataset

## Scenario

After initial training, the model may benefit from optimization to improve its response quality and accuracy. Traditional approaches would require retraining the model with additional data, which is time-consuming and computationally expensive. ZTOM provides an alternative solution that optimizes the model's performance using only a small validation set and semantic similarity metrics to guide the optimization process.

## Use Case

This demonstration is applicable when:
- A model is already trained and deployed, but performance improvements are desired
- Retraining is not feasible due to time, computational, or data constraints
- You have access to only a small validation set for optimization
- You want to improve response quality using semantic similarity as a guide

## Summary

This notebook demonstrates how the Zero-Train Optimization & Maintenance (ZTOM) improves the Medical Chatbot's response quality without requiring any retraining. Using semantic similarity to guide optimization, ZTOM reduced the model's error rate by approximately 1.37%, demonstrating measurable improvements in response quality with minimal data requirements.
  
## ZTOM Result Outputs

  | Metric                       | Value                          |
  |------------------------------|-------------------------------:|
  | Non-Optimized Model Error    | 0.387                          |
  | Optimized Model Error        | 0.350                          |
  | Improvement                  | ~1.37%                         |
  | Scaling Factors              | [-0.089,-0.173,-0.175,-0.199]  |
  | Number of Inferences         | 60                             |

### Install dependencies and utility functions: This cell should be run once.

In [ ]:
%%capture

%pip install 'authentrics==0.21.2' --extra-index-url='https://us-central1-python.pkg.dev/authentrics/authentrics/simple'

from authentrics import AuthentricsSession
from dotenv import load_dotenv
from transformers import Tokenizer


load_dotenv()

def cleanup_project(session: AuthentricsSession, project_name: str):
    projects = session.get_projects()
    for project in projects:
        if project.name == project_name:
            session.delete_project(project)
            return


def format_row(tokenizer: Tokenizer, example):
    messages = [
        {
            "role": "user",
            "content": (
                "Answer the legal question based on the contract.\n\n"
                f"Question: {example['question']}\n\n"
                f"Context:\n{example['context']}"
            ),
        },
        {"role": "assistant", "content": example["answer"]},
    ]
    example["formatted_chat"] = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return example


## Download Checkpoints and Dataset

In [8]:
from pathlib import Path

checkpoint_dir = Path('checkpoints')
data_dir = Path('data')

checkpoint_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

checkpoints = [checkpoint_dir / "cuad_checkpoints" / f"checkpoint-{i}" for i in range(300, 1201, 300)]
checkpoints.append(checkpoint_dir / "cuad_checkpoints" / "final_model")
checkpoints += [checkpoint_dir / "bill_sum_v2" / f"checkpoint-{i}" for i in range(300, 1201, 300)]
checkpoints.append(checkpoint_dir / "bill_sum_v2" / "final_model")

data_file = data_dir / "test_cuad_qa.json"

## Authentrics Python Library

In [14]:
from authentrics import AuthentricsException, AuthentricsSession, ZtomOptimizationOptions

PROJECT_NAME = "ZTOM_Local"
project_path = Path(PROJECT_NAME)

session = AuthentricsSession()
session.login()

try:
    project = session.load_project(project_path)

except AuthentricsException:
    cleanup_project(session, PROJECT_NAME)
    project = session.create_project(project_path, PROJECT_NAME)

    project = session.add_checkpoints(project, *checkpoints)


[DEBUG] Status code: 200
[INFO] Logged in successfully
[DEBUG] Making GET request to /project
[DEBUG] Status code: 200
[DEBUG] Making DELETE request to /project
[DEBUG] Body: {"hardDelete":false,"projectIds":["69c402e9690c8e290bed8593"]}
[DEBUG] Status code: 200
[DEBUG] Making POST request to /project
[DEBUG] Body: {"description":"","format":"ONNX","name":"ZTOM_Local","type":"CLIENT_MANAGED"}
[DEBUG] Status code: 200
[DEBUG] Creating checkpoint for project 69c40348690c8e290bed859f with filename checkpoint-300 and hash 58271e6bd4d0f6e1a1bd906b6bff5b0f1800e6766e15156026601f623b24fcc1
[DEBUG] Making POST request to /project/file/external
[DEBUG] Body: {"fileName":"checkpoint-300","filePath":"LOCAL","format":"ONNX","hash":"58271e6bd4d0f6e1a1bd906b6bff5b0f1800e6766e15156026601f623b24fcc1","projectId":"69c40348690c8e290bed859f"}
[DEBUG] Status code: 200
[DEBUG] Successfully created checkpoint: {"id":"69c40348690c8e290bed859f","name":"ZTOM_Local","description":"","format":"ONNX","createdOn":1

In [12]:
project

Project(id=69c402e9690c8e290bed8593, name='ZTOM_Local', description='', created_at='Wed Mar 25 15:44:41.951000000 2026', project_path='optional("/home/mike/demos/ZTOM_Local/ZTOM_Local")')

In [13]:
project.checkpoints

[Checkpoint(id=69c402e9690c8e290bed8595, name='checkpoint-300', hash=58271e6bd4d0f6e1a1bd906b6bff5b0f1800e6766e15156026601f623b24fcc1, created_at='Wed Mar 25 15:44:41.975000000 2026', path='optional("/home/mike/demos/ZTOM_Local/checkpoints/cuad_checkpoints/checkpoint-300")'),
 Checkpoint(id=69c402e9690c8e290bed8596, name='checkpoint-600', hash=6555d5a9df97f95368fbee731a797a3f74e1e81288eba44718e59a69af4359e6, created_at='Wed Mar 25 15:44:41.993000000 2026', path='optional("/home/mike/demos/ZTOM_Local/checkpoints/cuad_checkpoints/checkpoint-600")'),
 Checkpoint(id=69c402ea690c8e290bed8597, name='checkpoint-900', hash=ba95fa6a4b5892f36f94b554cd2a7f3b110f1b8ae4c7c21903ba2929aafa6ada, created_at='Wed Mar 25 15:44:42.011000000 2026', path='optional("/home/mike/demos/ZTOM_Local/checkpoints/cuad_checkpoints/checkpoint-900")'),
 Checkpoint(id=69c402ea690c8e290bed8598, name='checkpoint-1200', hash=3a1a5d5dd8415d656acfc18675a850f50def3185135245c6ed46d9b07c8ad1b7, created_at='Wed Mar 25 15:44:42.0

### Model Wrapper

In [ ]:
from authentrics import InferenceResult, ModelInterface, Parameters, WeightBias, use_backend
from datasets import Dataset
from transformers import TextGenerationPipeline
from transformers.pipelines import pipeline


class SimpleHFModel(ModelInterface):
    def __init__(self, dataset: Dataset | None = None, batch_size: int = 1):
        super().__init__()

        use_backend("torch")
        
        self._dataset = dataset or Dataset.from_list([])
        self._batch_size = batch_size

        self._inference_config = {"max_new_tokens": 50, "do_sample": False, "return_full_text": False}
        
        self._module = None
        self._input_data = None

    def load(self, checkpoint_path: Path | str | bytes) -> None:
        checkpoint_path = Path(checkpoint_path)

        self._module: TextGenerationPipeline = pipeline(
            "text-generation",
            model=str(checkpoint_path),
            trust_remote_code=True,
            device_map="sequential",
        )

        if self._input_data is None:
            self._input_data: list[str] = self._dataset.map(
                lambda example: format_row(self._module.tokenizer, example)
            )["formatted_chat"]

    def get_weight_bias(
        self,
        weight_names: list[str] | None = None,
        bias_names: list[str] | None = None,
    ) -> WeightBias:
        weights = Parameters()
        biases = Parameters()
        for name, param in self._module.model.named_parameters():
            last_part = name.rsplit(".", 1)[-1]
            if last_part == "weight":
                if weight_names is None or name in weight_names:
                    weights[name] = param.detach().cpu()

            elif last_part == "bias":
                if bias_names is None or name in bias_names:
                    biases[name] = param.detach().cpu()

        return WeightBias(weights, biases)

    def perform_inference(
        self,
        return_intermediate_outputs: bool = False,
        layer_names: list[str] | None = None,
    ) -> InferenceResult:
        chat_template = self._inference_config.pop("chat_template", None)

        # Perform inference
        assert self._module.tokenizer is not None
        if chat_template is not None:
            self._module.tokenizer.chat_template = chat_template

        result = self._module(
            text_inputs=self._input_data,
            batch_size=self._batch_size,
            chat_template=chat_template,
            **self._inference_config,
        )

        return InferenceResult(
            output=[r[0]["generated_text"] for r in result],
            intermediate_outputs=Parameters(),
        )

    def set_weight_bias(self, weight_bias: WeightBias) -> None:
        state = {n: p for n, p in self._module.model.named_parameters()}
        for name, tensor in weight_bias.weights.items():
            if name in state:
                state[name].data.copy_(tensor)

        for name, tensor in weight_bias.biases.items():
            if name in state:
                state[name].data.copy_(tensor)

    def save(self, checkpoint_path: Path | str | bytes) -> None:
        path = Path(checkpoint_path)
        path.parent.mkdir(parents=True, exist_ok=True)

        # Avoid a bug in the Hugging Face pipeline
        if not hasattr(self._module, "modelcard"):
            self._module.modelcard = None

        self._module.save_pretrained(path)


### Prepare the Data

In [16]:
from datasets import load_dataset


dataset = load_dataset(
    "json",
    data_files={"eval": str(data_file)},
)["eval"].take(100)

Generating eval split: 0 examples [00:00, ? examples/s]

### Run ZTOM Model Maintenance

#### Setup loss function

This is a custom loss function that calculates the average fuzzy match between the predicted and actual answers. We will maximize this function over the course of the optimization process.

In [ ]:
import torch
from rapidfuzz import fuzz


def clean_answer(text: str) -> str:
    if "<start_of_turn>model" in text:
        text = text.split("<start_of_turn>model")[-1]

    return text.strip()


def average_fuzzy_match(y_pred: list[str], answers: list[str]) -> float:
    if len(y_pred) != len(answers):
        raise ValueError("y_pred and answers must have the same length")

    total = len(y_pred)
    if total == 0:
        return 0.0

    scores = torch.tensor(
        [
            fuzz.partial_ratio(clean_answer(output).lower(), answer.lower())
            for output, answer in zip(y_pred, answers)
        ]
    )

    return scores.mean().item()


In [ ]:
session.model = SimpleHFModel(dataset=dataset, batch_size=10)

options = ZtomOptimizationOptions(
    max_evaluations=100,
    xtol_rel=1e-4,
    ftol_rel=1e-4,
    lower_bound=-1.0,
    upper_bound=1.0,
    minimize=False,
)

ztom_result = session.ztom_analysis(
    project,
    lambda y_pred: average_fuzzy_match(y_pred, dataset["answer"]),
    project_path / "optimized_checkpoint",
    options,
)

print(ztom_result)